Plot the per-year distribution of deeds across the corpus:

In [1]:
import numpy as np
import re
from pathlib import Path
from collections import Counter, defaultdict
from datetime import date

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

EMBEDDINGS_FILE = "../data/embeddings/embeddings_20-03-2026.npz"
IMAGES_DIR = "../data/deeds-img"
OUTPUT_FILE = "../outputs/deeds_frequency.jpg"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "font.sans-serif": ['Roboto Condensed'],
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "figure.facecolor": "white",
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.15,
})

emb_path = Path(EMBEDDINGS_FILE)
if emb_path.exists():
    data = np.load(EMBEDDINGS_FILE)
    filenames = [str(fn) for fn in data["filenames"]]
    print(f"Loaded {len(filenames)} filenames from {EMBEDDINGS_FILE}")
else:
    images_dir = Path(IMAGES_DIR)
    filenames = [f.name for f in images_dir.iterdir() if f.is_file()]
    print(f"Loaded {len(filenames)} filenames from {IMAGES_DIR}")


DATE_RE = re.compile(
    r'(?:^0-)?'           # optional "0-" embedding prefix (for the pretraining code)
    r'\d+_'               # document index (random)
    r'(\d{1,2})_'         # day
    r'(\d{1,2})_'         # month
    r'(\d{4})'            # year
)

parsed_dates = []
unparsed = []
for fn in filenames:
    m = DATE_RE.match(fn)
    if m:
        dd, mm, yyyy = int(m.group(1)), int(m.group(2)), int(m.group(3))
        mm = max(1, min(12, mm))
        dd = max(1, min(28, dd))
        parsed_dates.append((date(yyyy, mm, dd), fn))
    else:
        unparsed.append(fn)

print(f"Parsed {len(parsed_dates)} dates, {len(unparsed)} unparsable")
if unparsed[:5]:
    print(f"  Sample unparsed: {unparsed[:5]}")

if not parsed_dates:
    print("No dates found — exiting.")
    exit(1)

dates_only = [d for d, _ in parsed_dates]
min_date = min(dates_only)
max_date = max(dates_only)
print(f"Date range: {min_date} → {max_date}")

# Print deeds sharing the exact same day (to report in main text)
day_to_files = defaultdict(list)
for d, fn in parsed_dates:
    day_to_files[d].append(fn)

same_day = {d: fns for d, fns in day_to_files.items() if len(fns) > 1}
if same_day:
    print(f"\n{'='*60}")
    print(f"Deeds sharing the exact same date  ({len(same_day)} dates)")
    print(f"{'='*60}")
    for d in sorted(same_day):
        print(f"\n  {d}  ({len(same_day[d])} deeds):")
        for fn in sorted(same_day[d]):
            print(f"    - {fn}")
else:
    print("\nNo deeds share the exact same date.")

yearly_counts = Counter(d.year for d in dates_only)
years = sorted(yearly_counts.keys())
counts = [yearly_counts[y] for y in years]

fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(years, counts, color="black", width=0.8, edgecolor="black")

ax.set_xlabel("Year")
ax.set_ylabel("Number of deeds")
ax.set_title(
    f"Deed frequency distribution ({min_date.year}–{max_date.year}, "
    f"n = {len(dates_only)})"
)

if len(years) <= 25:
    ax.set_xticks(years)
    ax.set_xticklabels(years, rotation=45, ha="right")

ax.set_xlim(years[0] - 1, years[-1] + 1)

fig.savefig(OUTPUT_FILE)
plt.close(fig)

pdf_kb = Path(OUTPUT_FILE).stat().st_size / 1024
print(f"\nSaved {OUTPUT_FILE} ({pdf_kb:.1f} KB)")

Loaded 470 filenames from ../data/embeddings/embeddings_20-03-2026.npz
Parsed 466 dates, 4 unparsable
  Sample unparsed: ['0-0447_XX_XX_1327-01.png', '0-0448_XX_XX_1348-01.png', '0-0449_XX_XX_1348-02.png', '0-0450_XX_XX_1347-01.png']
Date range: 1300-01-10 → 1357-07-22

Deeds sharing the exact same date  (26 dates)

  1306-11-20  (2 deeds):
    - 0-0317_20_11_1306-01.png
    - 0-0318_20_11_1306-02.png

  1306-12-21  (2 deeds):
    - 0-0326_21_12_1306-01.png
    - 0-0327_21_12_1306-02.png

  1307-03-06  (2 deeds):
    - 0-0089_06_03_1307-01.png
    - 0-0090_06_03_1307-02.png

  1307-11-18  (2 deeds):
    - 0-0287_18_11_1307-01.png
    - 0-0288_18_11_1307-02.png

  1310-10-01  (2 deeds):
    - 0-0023_01_10_1310-01.png
    - 0-0024_01_10_1310-02.png

  1327-08-08  (3 deeds):
    - 0-0465_08_08_1327-01.png
    - 0-0466_08_08_1327-02.png
    - 0-0467_08_08_1327-03.png

  1328-04-08  (2 deeds):
    - 0-0120_08_04_1328-01.png
    - 0-0121_08_04_1328-02.png

  1328-05-08  (2 deeds):
    - 0-01